In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
files = glob.glob("../data/*.csv")

print("Number of files:", len(files))

for file in files:
    print(file)

Number of files: 8
../data\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
../data\Friday-WorkingHours-Afternoon.csv
../data\Friday-WorkingHours-Morning.pcap_ISCX.csv
../data\Monday-WorkingHours.pcap_ISCX.csv
../data\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
../data\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
../data\Tuesday-WorkingHours.pcap_ISCX.csv
../data\Wednesday-workingHours.pcap_ISCX.csv


Now firstly we will be cleaning one file first, to check whether it is working properly then we will be applying to our rest 7 csv files.

In [3]:
df = pd.read_csv(files[0])

print(df.shape)

(286467, 79)


In [4]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

In [5]:
numeric_cols = df.select_dtypes(include="number").columns

infinite_counts = np.isinf(df[numeric_cols]).sum()

print(infinite_counts[infinite_counts > 0])

Flow Bytes/s      356
Flow Packets/s    371
dtype: int64


In [6]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [8]:
print("Infinite values after replacement:",
      np.isinf(df.select_dtypes(include="number")).sum().sum())

print("Missing values after replacement:",
      df.isnull().sum().sum())

Infinite values after replacement: 0
Missing values after replacement: 742


In [9]:
missing_counts = df.isnull().sum()

print(missing_counts[missing_counts > 0])

Flow Bytes/s      371
Flow Packets/s    371
dtype: int64


In [10]:
numeric_columns = df.select_dtypes(include="number").columns

df[numeric_columns] = df[numeric_columns].fillna(
    df[numeric_columns].median()
)

Done with putting column median value for missing values

In [11]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 0


In [12]:
duplicate_rows = df[df.duplicated(keep=False)]

print("Total duplicate rows:", len(duplicate_rows))

Total duplicate rows: 107704


In [15]:
print("Unique labels among duplicate rows:")
print(duplicate_rows["Label"].value_counts())#tells us what kinds of traffic are involved in the duplicated records.

Unique labels among duplicate rows:
Label
PortScan    101501
BENIGN        6203
Name: count, dtype: int64


See whether the same features ever have different labels

In [16]:
feature_columns = [col for col in df.columns if col != "Label"]

duplicate_features = df[df.duplicated(
    subset=feature_columns,
    keep=False
)]

print("Rows with duplicate features:", len(duplicate_features))

print(
    duplicate_features.groupby(feature_columns)["Label"]
    .nunique()
    .value_counts()
)

Rows with duplicate features: 107704
Label
1    35351
Name: count, dtype: int64


In [17]:
label_conflicts = (
    duplicate_features
    .groupby(feature_columns)["Label"]
    .nunique()
)

print("Feature groups with different labels:",
      (label_conflicts > 1).sum())

Feature groups with different labels: 0


Removing Duplicates becuase no duplicates have diffrent Label (attack type)

In [18]:
print("Before removing duplicates:", df.shape)

df = df.drop_duplicates()

print("After removing duplicates:", df.shape)

Before removing duplicates: (286467, 79)
After removing duplicates: (214114, 79)


In [19]:
print(df["Label"].value_counts())

Label
BENIGN      123295
PortScan     90819
Name: count, dtype: int64


In [21]:
def clean_data(file):

    print("\nCleaning:", file)

    # 1. Load CSV
    df = pd.read_csv(file)

    # 2. Remove extra spaces from column names
    df.columns = df.columns.str.strip()

    # 3. Replace infinity with NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 4. Fill missing numerical values with median
    numeric_columns = df.select_dtypes(include="number").columns
    df[numeric_columns] = df[numeric_columns].fillna(
        df[numeric_columns].median()
    )

    # 5. Remove exact duplicate rows
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)

    print("Rows before duplicates:", before)
    print("Rows after duplicates:", after)
    print("Rows removed:", before - after)

    return df

In [23]:
test_df = clean_data(files[0])

print("\nFinal shape:", test_df.shape)


Cleaning: ../data\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Rows before duplicates: 286467
Rows after duplicates: 214114
Rows removed: 72353

Final shape: (214114, 79)


Making Clean folder to store data

In [24]:
cleaned_dir = "../data/cleaned"

os.makedirs(cleaned_dir, exist_ok=True)

print("Cleaned folder ready:", cleaned_dir)

Cleaned folder ready: ../data/cleaned


In [25]:
for file in files:

    clean_df = clean_data(file)

    filename = os.path.basename(file)

    output_file = os.path.join(
        cleaned_dir,
        filename.replace(".csv", "_cleaned.csv")
    )

    clean_df.to_csv(output_file, index=False)

    print("Saved:", output_file)

    del clean_df


Cleaning: ../data\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Rows before duplicates: 286467
Rows after duplicates: 214114
Rows removed: 72353
Saved: ../data/cleaned\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.csv

Cleaning: ../data\Friday-WorkingHours-Afternoon.csv
Rows before duplicates: 225745
Rows after duplicates: 223112
Rows removed: 2633
Saved: ../data/cleaned\Friday-WorkingHours-Afternoon_cleaned.csv

Cleaning: ../data\Friday-WorkingHours-Morning.pcap_ISCX.csv
Rows before duplicates: 191033
Rows after duplicates: 184145
Rows removed: 6888
Saved: ../data/cleaned\Friday-WorkingHours-Morning.pcap_ISCX_cleaned.csv

Cleaning: ../data\Monday-WorkingHours.pcap_ISCX.csv
Rows before duplicates: 529918
Rows after duplicates: 502983
Rows removed: 26935
Saved: ../data/cleaned\Monday-WorkingHours.pcap_ISCX_cleaned.csv

Cleaning: ../data\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Rows before duplicates: 288602
Rows after duplicates: 252972
Rows removed